[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Chatterbox_Colab.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Chatterbox_Colab.ipynb)

# 🎙️ Chatterbox & Chatterbox-Turbo — Zero-Shot Voice Cloning, Emotion Exaggeration, Paralinguistic Tags, 23-Language TTS & Voice Conversion

> **Runtime required:** GPU — **T4 (16 GB)**, **L4 (24 GB)**, or **A100 (40/80 GB)**. All four official Chatterbox models fit comfortably on a free Colab T4 GPU (~4–6 GB VRAM per active engine).

A complete Google Colab studio for **[Resemble AI Chatterbox](https://github.com/resemble-ai/chatterbox)** (**MIT License**) — an open-source family of production-grade Text-to-Speech (TTS) and Voice Conversion (VC) models featuring **first-of-its-kind Emotion Exaggeration control**, **1-step MeanFlow distilled Turbo synthesis with native paralinguistic tags**, **23-language zero-shot cross-lingual voice cloning**, and **Perth implicit watermarking**.

**Upstream links:** [resemble-ai/chatterbox](https://github.com/resemble-ai/chatterbox) · [ResembleAI/chatterbox](https://huggingface.co/ResembleAI/chatterbox) · [ResembleAI/chatterbox-turbo](https://huggingface.co/ResembleAI/chatterbox-turbo) · [HF Space](https://huggingface.co/spaces/ResembleAI/Chatterbox) · [MIT License](https://github.com/resemble-ai/chatterbox/blob/master/LICENSE)

---

## 🧩 All 4 Official Chatterbox Engines in One Notebook

| Engine Class | HF Checkpoint | Key Capabilities | Output SR | Typical VRAM |
|---|---|---|---|---|
| **`ChatterboxTTS`** (`chatterbox.tts`) | `ResembleAI/chatterbox` | Zero-shot voice cloning (5–15s prompt) + **Emotion Exaggeration (`0.0–2.0`)** + **`cfg_weight` (`0.0–1.0`)** pacing control | 24 kHz | ~4.8 GB |
| **`ChatterboxTurboTTS`** (`chatterbox.tts_turbo`) | `ResembleAI/chatterbox-turbo` / `chatterbox-nano` | **1-step MeanFlow distilled S3Gen decoder** + native **Paralinguistic Event Tags** (`[laugh]`, `[chuckle]`, `[cough]`, `[sigh]`, `[gasp]`, `[clear throat]`, `[groan]`, `[sniff]`, `[shush]`) + LUFS loudness normalization | 24 kHz | ~3.8 GB (Turbo) / ~2.2 GB (Nano) |
| **`ChatterboxMultilingualTTS`** (`chatterbox.mtl_tts`) | `ResembleAI/chatterbox` (`v2` & `v3`) | **23 languages** (`ar, da, de, el, en, es, fi, fr, he, hi, it, ja, ko, ms, nl, no, pl, pt, ru, sv, sw, tr, zh`) with zero-shot cross-lingual voice transfer & emotion control | 24 kHz | ~5.2 GB |
| **`ChatterboxVC`** (`chatterbox.vc`) | `ResembleAI/chatterbox` | Audio-to-audio **Zero-Shot Voice Conversion** — preserves source speech content/timing while transferring target speaker timbre via `S3Gen` | 24 kHz | ~2.5 GB |

## 🔗 Cross-Notebook Avatar & Video Integration

- Every generated `.wav` file is automatically saved to `/content/drive/MyDrive/AEI_TTS_Out/Chatterbox/`.
- **`InfiniteTalk_Colab.ipynb`** and **`LongCat-Video-Avatar-1.5_Colab.ipynb`** automatically discover files in `/content/drive/MyDrive/AEI_TTS_Out/Chatterbox/` via their **📂 Cross-Notebook Drive Audio Picker**.
- Includes a built-in **📂 Cross-Notebook Drive Audio & Voice Picker** to load reference voices from `/content/drive/MyDrive/AEI_TTS_Cache/voices/` or source clips from other AEI notebooks.

## 🚀 Quick Start

1. **STEP 1** — Mount Google Drive & install `chatterbox-tts` + `resemble-perth` + audio dependencies (~2–3 min).
2. **STEP 2** — Pre-cache `ResembleAI/chatterbox` and `ResembleAI/chatterbox-turbo` weights + starter reference voices to Drive.
3. **STEP 3** — Load the lazy multi-engine synthesis & voice conversion core functions.
4. **STEP 4** — Launch the 4-Tab Gradio Studio UI.
5. **STEP 5** — Optional Keep-Alive heartbeat for long sessions.
6. **STEP 6** — Form-based Quick Test with inline player and `FileLink` download.
7. **STEP 7** — Batch Audiobook / Multi-Speaker Script generator with `tqdm` and per-item fault tolerance.

In [ ]:
# @title STEP 1 — Install Chatterbox Dependencies & Configure Drive Cache
# @markdown Mounts Google Drive, configures `HF_HOME` at `/content/drive/MyDrive/AEI_TTS_Cache/Chatterbox`, and installs `chatterbox-tts` (`resemble-ai/chatterbox`), `resemble-perth`, `s3tokenizer`, and pinned `gradio>=5.0.0`.

import os, sys, subprocess, shutil, time, pathlib
import torch

print("=" * 72)
print("🎙️ Chatterbox Studio — Environment Setup & Installation")
print("=" * 72)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GPU] PyTorch {torch.__version__} · Device = {DEVICE}", flush=True)
if DEVICE == "cuda":
    try:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[GPU] {gpu_name} ({vram_gb:.1f} GB VRAM)", flush=True)
    except Exception as e:
        print(f"[GPU] Could not query GPU properties: {e}", flush=True)

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    try:
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[Drive] Notice — running with local storage fallback ({e})", flush=True)

if os.path.exists("/content/drive/MyDrive"):
    CACHE_ROOT = "/content/drive/MyDrive/AEI_TTS_Cache/Chatterbox"
    OUT_DIR = "/content/drive/MyDrive/AEI_TTS_Out/Chatterbox"
    VOICES_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices"
else:
    CACHE_ROOT = "/content/cache/Chatterbox"
    OUT_DIR = "/content/outputs/Chatterbox"
    VOICES_DIR = "/content/cache/voices"

for d in (CACHE_ROOT, OUT_DIR, VOICES_DIR, "/content/chatterbox_out"):
    os.makedirs(d, exist_ok=True)

os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hub")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"[Cache] HF_HOME     = {CACHE_ROOT}", flush=True)
print(f"[Out]   OUT_DIR     = {OUT_DIR}", flush=True)
print(f"[Voice] VOICES_DIR  = {VOICES_DIR}", flush=True)

if not shutil.which("ffmpeg"):
    print("[apt] Installing ffmpeg...", flush=True)
    subprocess.run(["apt-get", "-qq", "-y", "install", "ffmpeg"], check=False)

t0 = time.time()
print("\n[pip] Installing resemble-ai/chatterbox (--no-deps to preserve Colab PyTorch)...", flush=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "git+https://github.com/resemble-ai/chatterbox.git",
        "git+https://github.com/resemble-ai/Perth.git",
    ],
    check=True,
)

print("[pip] Installing runtime dependencies (s3tokenizer, conformer, pyloudnorm, gradio>=5.0.0, tqdm)...", flush=True)
RUNTIME_PKGS = [
    "s3tokenizer",
    "conformer==0.3.2",
    "spacy-pkuseg",
    "pykakasi==2.3.0",
    "pyloudnorm",
    "omegaconf",
    "librosa>=0.10.2",
    "soundfile",
    "scipy",
    "safetensors>=0.4.5",
    "diffusers>=0.29.0",
    "transformers>=4.46.0",
    "accelerate>=0.34.0",
    "huggingface_hub>=0.26.0",
    "tqdm>=4.66.0",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + RUNTIME_PKGS, check=True)
print(f"[pip] Packages installed in {time.time() - t0:.1f}s", flush=True)

print("\n[Verify] Checking Chatterbox module imports...", flush=True)
try:
    from chatterbox.tts import ChatterboxTTS
    from chatterbox.tts_turbo import ChatterboxTurboTTS
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES
    from chatterbox.vc import ChatterboxVC
    print(f"  ✓ ChatterboxTTS, ChatterboxTurboTTS, ChatterboxMultilingualTTS ({len(SUPPORTED_LANGUAGES)} langs), ChatterboxVC imported OK!")
except Exception as e:
    print(f"  ✗ Import verification failed: {e}")
    raise

print("\n[Done] STEP 1 complete — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Cache Chatterbox, Chatterbox-Turbo & Multilingual Checkpoints
# @markdown Downloads checkpoints for `ResembleAI/chatterbox` (Standard TTS, Voice Conversion, and 23-Language Multilingual TTS) and `ResembleAI/chatterbox-turbo` (1-step MeanFlow Distilled Turbo TTS) into `/content/drive/MyDrive/AEI_TTS_Cache/Chatterbox`, plus sample reference voice clips.

import os, time, urllib.request
from pathlib import Path
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
import huggingface_hub.constants as hf_constants

CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/Chatterbox")
os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT
os.makedirs(CACHE_ROOT, exist_ok=True)

# Disable Xet backend if buggy on certain Colab huggingface_hub versions
hf_constants.HF_HUB_DISABLE_XET = True

PRECACHE_STANDARD_AND_VC = True  # @param {type:"boolean"}
PRECACHE_TURBO = True  # @param {type:"boolean"}
PRECACHE_MULTILINGUAL = True  # @param {type:"boolean"}
MULTILINGUAL_VERSION = "v2"  # @param ["v2", "v3"]

def _safe_snapshot(repo_id: str, allow_patterns: list, label: str):
    print(f"\n[{label}] Caching {repo_id} -> {CACHE_ROOT} ...", flush=True)
    t0 = time.time()
    try:
        path = snapshot_download(
            repo_id=repo_id,
            repo_type="model",
            revision="main",
            allow_patterns=allow_patterns,
        )
        size_mb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fs in os.walk(path)
            for f in fs
            if os.path.exists(os.path.join(dp, f))
        ) / (1024 ** 2)
        print(f"  ✓ [{label}] Ready at {path} ({size_mb:.1f} MB in {time.time() - t0:.1f}s)", flush=True)
        return path
    except Exception as e:
        print(f"  ⚠ [{label}] Warning during pre-cache ({e}) — will download on demand in STEP 3.", flush=True)
        return None

if PRECACHE_STANDARD_AND_VC:
    _safe_snapshot(
        "ResembleAI/chatterbox",
        ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "tokenizer.json", "conds.pt"],
        "Chatterbox-TTS & VC",
    )

if PRECACHE_TURBO:
    _safe_snapshot(
        "ResembleAI/chatterbox-turbo",
        ["*.safetensors", "*.json", "*.txt", "*.pt", "*.model"],
        "Chatterbox-Turbo",
    )

if PRECACHE_MULTILINGUAL:
    t3_ckpt = "t3_mtl23ls_v3.safetensors" if MULTILINGUAL_VERSION == "v3" else "t3_mtl23ls_v2.safetensors"
    _safe_snapshot(
        "ResembleAI/chatterbox",
        ["ve.pt", t3_ckpt, "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"],
        f"Chatterbox-Multilingual ({MULTILINGUAL_VERSION})",
    )

# Download official Resemble AI demo reference voices into AEI_TTS_Cache/voices
VOICES_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices" if os.path.exists("/content/drive/MyDrive") else "/content/cache/voices"
os.makedirs(VOICES_DIR, exist_ok=True)

DEMO_VOICES = {
    "chatterbox_female_podcast.wav": "https://storage.googleapis.com/chatterbox-demo-samples/prompts/female_random_podcast.wav",
    "chatterbox_en_female.flac": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
    "chatterbox_fr_female.flac": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/fr_f1.flac",
    "chatterbox_es_female.flac": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/es_f1.flac",
    "chatterbox_ja_prompt.flac": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ja/ja_prompts1.flac",
}

print("\n[Voices] Ensuring starter reference voices in", VOICES_DIR)
for fname, url in tqdm(DEMO_VOICES.items(), desc="Reference voices"):
    dest = os.path.join(VOICES_DIR, fname)
    if not os.path.exists(dest) or os.path.getsize(dest) < 1024:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=20) as r, open(dest, "wb") as f:
                f.write(r.read())
            print(f"  ✓ Downloaded {fname}")
        except Exception as e:
            print(f"  ⚠ Could not fetch {fname}: {e}")
    else:
        print(f"  ✓ Cached {fname}")

print("\n[Done] STEP 2 complete — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core Synthesis & Voice Conversion Functions (Lazy Multi-Engine Loader)
# @markdown Defines the lazy model manager and synthesis pipelines for all 4 Chatterbox engines (`ChatterboxTTS`, `ChatterboxTurboTTS`, `ChatterboxMultilingualTTS`, and `ChatterboxVC`), plus Cross-Notebook Drive Audio discovery and automatic sentence-chunked long-form generation.

import os, re, gc, time, random, zipfile, builtins
from pathlib import Path
from typing import Optional, List, Dict, Tuple
import numpy as np
import torch
import soundfile as sf
from tqdm.auto import tqdm

from chatterbox.tts import ChatterboxTTS
from chatterbox.tts_turbo import ChatterboxTurboTTS
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES
from chatterbox.vc import ChatterboxVC

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/Chatterbox")
os.environ.setdefault("HF_HOME", CACHE_ROOT)

OUT_DIR = "/content/drive/MyDrive/AEI_TTS_Out/Chatterbox" if os.path.exists("/content/drive/MyDrive") else "/content/outputs/Chatterbox"
VOICES_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices" if os.path.exists("/content/drive/MyDrive") else "/content/cache/voices"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)
os.makedirs("/content/chatterbox_out", exist_ok=True)

AUTO_SWAP_ENGINES_FOR_VRAM = True  # @param {type:"boolean"}
# @markdown When `AUTO_SWAP_ENGINES_FOR_VRAM = True`, loading a new Chatterbox engine automatically unloads any previously loaded engine first so T4 (16 GB) GPUs never run out of VRAM.

TURBO_EVENT_TAGS = [
    "[laugh]", "[chuckle]", "[cough]", "[sigh]", "[gasp]",
    "[clear throat]", "[groan]", "[sniff]", "[shush]",
]

MULTILINGUAL_SAMPLE_TEXTS = {
    "ar": "في الشهر الماضي، وصلنا إلى معلم جديد بمليارين من المشاهدات على قناتنا على يوتيوب.",
    "da": "Sidste måned nåede vi en ny milepæl med to milliarder visninger på vores YouTube-kanal.",
    "de": "Letzten Monat haben wir einen neuen Meilenstein erreicht: zwei Milliarden Aufrufe auf unserem YouTube-Kanal.",
    "el": "Τον περασμένο μήνα, φτάσαμε σε ένα νέο ορόσημο με δύο δισεκατομμύρια προβολές στο κανάλι μας στο YouTube.",
    "en": "Last month, we reached a new milestone with two billion views on our YouTube channel.",
    "es": "El mes pasado alcanzamos un nuevo hito: dos mil millones de visualizaciones en nuestro canal de YouTube.",
    "fi": "Viime kuussa saavutimme uuden virstanpylvään kahden miljardin katselukerran kanssa YouTube-kanavallamme.",
    "fr": "Le mois dernier, nous avons atteint un nouveau jalon avec deux milliards de vues sur notre chaîne YouTube.",
    "he": "בחודש שעבר הגענו לאבן דרך חדשה עם שני מיליארד צפיות בערוץ היוטיוב שלנו.",
    "hi": "पिछले महीने हमने एक नया मील का पत्थर छुआ: हमारे YouTube चैनल पर दो अरब व्यूज़।",
    "it": "Il mese scorso abbiamo raggiunto un nuovo traguardo: due miliardi di visualizzazioni sul nostro canale YouTube.",
    "ja": "先月、私たちのYouTubeチャンネルで二十億回の再生回数という新たなマイルストーンに到達しました。",
    "ko": "지난달 우리는 유튜브 채널에서 이십억 조회수라는 새로운 이정표에 도달했습니다.",
    "ms": "Bulan lepas, kami mencapai pencapaian baru dengan dua bilion tontonan di saluran YouTube kami.",
    "nl": "Vorige maand bereikten we een nieuwe mijlpaal met two miljard weergaven op ons YouTube-kanaal.",
    "no": "Forrige måned nådde vi en ny milepæl med to milliarder visninger på YouTube-kanalen vår.",
    "pl": "W zeszłym miesiącu osiągnęliśmy nowy kamień milowy z dwoma miliardami wyświetleń na naszym kanale YouTube.",
    "pt": "No mês passado, alcançámos um novo marco: dois mil milhões de visualizações no nosso canal do YouTube.",
    "ru": "В прошлом месяце мы достигли нового рубежа: два миллиарда просмотров на нашем YouTube-канале.",
    "sv": "Förra månaden nådde vi en ny milstolpe med två miljarder visningar på vår YouTube-kanal.",
    "sw": "Mwezi uliopita, tulifika hatua mpya ya maoni ya bilioni mbili kweny kituo chetu cha YouTube.",
    "tr": "Geçen ay YouTube kanalımızda iki milyar görüntüleme ile yeni bir dönüm noktasına ulaştık.",
    "zh": "上个月，我们达到了一个新的里程碑。我们的YouTube频道观看次数达到了二十亿次，这绝对令人难以置信。",
}

LOADED_ENGINES: Dict[str, object] = {}


def set_seed(seed: int) -> int:
    if seed is None or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    return seed


def free_chatterbox_vram(except_key: Optional[str] = None) -> str:
    keys = [k for k in list(LOADED_ENGINES.keys()) if k != except_key]
    for k in keys:
        LOADED_ENGINES.pop(k, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        free_b, total_b = torch.cuda.mem_get_info(0)
        return f"Active engines: {list(LOADED_ENGINES.keys()) or 'none'} · GPU free: {free_b/1e9:.1f}/{total_b/1e9:.1f} GB"
    return f"Active engines: {list(LOADED_ENGINES.keys()) or 'none'} (CPU)"


def get_engine(engine_key: str):
    """Lazily load one of: 'tts', 'turbo', 'nano', 'mtl_v2', 'mtl_v3', 'vc'."""
    if engine_key in LOADED_ENGINES and LOADED_ENGINES[engine_key] is not None:
        return LOADED_ENGINES[engine_key]
    if AUTO_SWAP_ENGINES_FOR_VRAM and LOADED_ENGINES:
        free_chatterbox_vram(except_key=engine_key)
    t0 = time.time()
    print(f"[Engine] Loading '{engine_key}' on {DEVICE} ...", flush=True)
    if engine_key == "tts":
        model = ChatterboxTTS.from_pretrained(device=DEVICE)
    elif engine_key == "turbo":
        model = ChatterboxTurboTTS.from_pretrained(device=DEVICE, nano=False)
    elif engine_key == "nano":
        model = ChatterboxTurboTTS.from_pretrained(device=DEVICE, nano=True)
    elif engine_key == "mtl_v2":
        model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE, t3_model="v2")
    elif engine_key == "mtl_v3":
        model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE, t3_model="v3")
    elif engine_key == "vc":
        model = ChatterboxVC.from_pretrained(device=DEVICE)
    else:
        raise ValueError(f"Unknown Chatterbox engine_key: {engine_key!r}")
    LOADED_ENGINES[engine_key] = model
    print(f"[Engine] '{engine_key}' ready in {time.time() - t0:.1f}s (sr={model.sr} Hz)", flush=True)
    return model


def discover_aei_drive_audio(max_files: int = 60) -> List[str]:
    """Scan AEI Drive folders for reference voices and generated audio clips."""
    search_roots = [
        VOICES_DIR,
        OUT_DIR,
        "/content/drive/MyDrive/AEI_TTS_Out",
        "/content/drive/MyDrive/AEI_3D_Out",
    ]
    exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    found = []
    seen = set()
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dp, _, files in os.walk(root):
            for fn in files:
                if os.path.splitext(fn)[1].lower() in exts:
                    p = os.path.abspath(os.path.join(dp, fn))
                    if p in seen:
                        continue
                    seen.add(p)
                    try:
                        found.append((os.path.getmtime(p), p))
                    except OSError:
                        pass
    found.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in found[:max_files]]


def split_text_into_chunks(text: str, max_chars: int = 280) -> List[str]:
    """Split long paragraphs at sentence boundaries so each chunk stays within T3's optimal window."""
    text = re.sub(r"\s+", " ", (text or "").strip())
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    sentences = re.split(r"(?<=[.!?。！？])\s+", text)
    chunks, current = [], ""
    for sent in sentences:
        if not sent:
            continue
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip() if current else sent
        else:
            if current:
                chunks.append(current)
            if len(sent) <= max_chars:
                current = sent
            else:
                words = sent.split(" ")
                sub = ""
                for w in words:
                    if len(sub) + len(w) + 1 <= max_chars:
                        sub = f"{sub} {w}".strip() if sub else w
                    else:
                        if sub:
                            chunks.append(sub)
                        sub = w
                current = sub
    if current:
        chunks.append(current)
    return chunks


def save_chatterbox_wav(wav_tensor_or_np, sr: int, prefix: str = "chatterbox") -> Tuple[str, np.ndarray]:
    if torch.is_tensor(wav_tensor_or_np):
        wav_np = wav_tensor_or_np.detach().cpu().squeeze().numpy()
    else:
        wav_np = np.asarray(wav_tensor_or_np).squeeze()
    wav_np = np.asarray(wav_np, dtype=np.float32)
    wav_np = np.clip(wav_np, -1.0, 1.0)
    ts = int(time.time() * 1000)
    fname = f"{prefix}_{ts}.wav"
    drive_path = os.path.join(OUT_DIR, fname)
    sf.write(drive_path, wav_np, int(sr), subtype="PCM_16")
    local_copy = os.path.join("/content/chatterbox_out", fname)
    if os.path.abspath(drive_path) != os.path.abspath(local_copy):
        try:
            sf.write(local_copy, wav_np, int(sr), subtype="PCM_16")
        except Exception:
            pass
    return drive_path, wav_np


def synthesize_expressive_tts(
    text: str,
    audio_prompt_path: Optional[str] = None,
    exaggeration: float = 0.5,
    cfg_weight: float = 0.5,
    temperature: float = 0.8,
    min_p: float = 0.05,
    top_p: float = 1.0,
    repetition_penalty: float = 1.2,
    seed: int = 0,
    auto_chunk: bool = True,
    pause_sec: float = 0.18,
    prefix: str = "cb_tts",
) -> Dict[str, object]:
    """Run ChatterboxTTS with emotion exaggeration (0.0-2.0) and cfg_weight (0.0-1.0)."""
    model = get_engine("tts")
    seed_used = set_seed(seed)
    prompt_path = audio_prompt_path if (audio_prompt_path and os.path.exists(str(audio_prompt_path))) else None

    chunks = split_text_into_chunks(text, max_chars=280) if auto_chunk else [text.strip()]
    if not chunks:
        raise ValueError("Please provide non-empty text to synthesize.")

    t0 = time.time()
    pieces = []
    for idx, chunk in enumerate(chunks):
        wav_t = model.generate(
            chunk,
            audio_prompt_path=prompt_path if idx == 0 else None,
            exaggeration=float(exaggeration),
            cfg_weight=float(cfg_weight),
            temperature=float(temperature),
            min_p=float(min_p),
            top_p=float(top_p),
            repetition_penalty=float(repetition_penalty),
        )
        pieces.append(wav_t.detach().cpu().squeeze().numpy())
        if idx < len(chunks) - 1 and pause_sec > 0:
            pieces.append(np.zeros(int(model.sr * pause_sec), dtype=np.float32))

    full_wav = np.concatenate(pieces) if len(pieces) > 1 else pieces[0]
    out_path, wav_np = save_chatterbox_wav(full_wav, model.sr, prefix=prefix)
    elapsed = time.time() - t0
    dur = len(wav_np) / model.sr
    return {
        "wav_path": out_path,
        "audio_np": wav_np,
        "sr": model.sr,
        "duration_s": dur,
        "elapsed_s": elapsed,
        "seed_used": seed_used,
        "chunks": len(chunks),
    }


def synthesize_turbo_tts(
    text: str,
    audio_prompt_path: Optional[str] = None,
    nano: bool = False,
    temperature: float = 0.8,
    min_p: float = 0.0,
    top_p: float = 0.95,
    top_k: int = 1000,
    repetition_penalty: float = 1.2,
    norm_loudness: bool = True,
    seed: int = 0,
    auto_chunk: bool = True,
    pause_sec: float = 0.15,
    prefix: str = "cb_turbo",
) -> Dict[str, object]:
    """Run ChatterboxTurboTTS (1-step MeanFlow decoder + paralinguistic tags [laugh], [chuckle], [sigh], etc.)."""
    engine_key = "nano" if nano else "turbo"
    model = get_engine(engine_key)
    seed_used = set_seed(seed)

    prompt_path = audio_prompt_path if (audio_prompt_path and os.path.exists(str(audio_prompt_path))) else None
    if prompt_path is None:
        default_v = os.path.join(VOICES_DIR, "chatterbox_female_podcast.wav")
        if os.path.exists(default_v):
            prompt_path = default_v

    chunks = split_text_into_chunks(text, max_chars=280) if auto_chunk else [text.strip()]
    if not chunks:
        raise ValueError("Please provide non-empty text to synthesize.")

    t0 = time.time()
    pieces = []
    for idx, chunk in enumerate(chunks):
        wav_t = model.generate(
            chunk,
            audio_prompt_path=prompt_path if idx == 0 else None,
            temperature=float(temperature),
            min_p=float(min_p),
            top_p=float(top_p),
            top_k=int(top_k),
            repetition_penalty=float(repetition_penalty),
            norm_loudness=bool(norm_loudness),
        )
        pieces.append(wav_t.detach().cpu().squeeze().numpy())
        if idx < len(chunks) - 1 and pause_sec > 0:
            pieces.append(np.zeros(int(model.sr * pause_sec), dtype=np.float32))

    full_wav = np.concatenate(pieces) if len(pieces) > 1 else pieces[0]
    out_path, wav_np = save_chatterbox_wav(full_wav, model.sr, prefix=prefix)
    elapsed = time.time() - t0
    dur = len(wav_np) / model.sr
    return {
        "wav_path": out_path,
        "audio_np": wav_np,
        "sr": model.sr,
        "duration_s": dur,
        "elapsed_s": elapsed,
        "seed_used": seed_used,
        "chunks": len(chunks),
    }


def synthesize_multilingual_tts(
    text: str,
    language_id: str = "en",
    audio_prompt_path: Optional[str] = None,
    t3_version: str = "v2",
    exaggeration: float = 0.5,
    cfg_weight: float = 0.5,
    temperature: float = 0.8,
    repetition_penalty: float = 1.2,
    min_p: float = 0.05,
    top_p: float = 1.0,
    seed: int = 0,
    auto_chunk: bool = True,
    pause_sec: float = 0.18,
    prefix: str = "cb_mtl",
) -> Dict[str, object]:
    """Run ChatterboxMultilingualTTS across 23 supported languages."""
    lang_code = (language_id or "en").split(" ")[0].strip().lower()
    if lang_code not in SUPPORTED_LANGUAGES:
        raise ValueError(f"Unsupported language code '{lang_code}'. Supported: {sorted(SUPPORTED_LANGUAGES.keys())}")
    engine_key = "mtl_v3" if str(t3_version).lower() == "v3" else "mtl_v2"
    model = get_engine(engine_key)
    seed_used = set_seed(seed)
    prompt_path = audio_prompt_path if (audio_prompt_path and os.path.exists(str(audio_prompt_path))) else None

    chunks = split_text_into_chunks(text, max_chars=260) if auto_chunk else [text.strip()]
    if not chunks:
        raise ValueError("Please provide non-empty text to synthesize.")

    t0 = time.time()
    pieces = []
    for idx, chunk in enumerate(chunks):
        wav_t = model.generate(
            chunk,
            language_id=lang_code,
            audio_prompt_path=prompt_path if idx == 0 else None,
            exaggeration=float(exaggeration),
            cfg_weight=float(cfg_weight),
            temperature=float(temperature),
            repetition_penalty=float(repetition_penalty),
            min_p=float(min_p),
            top_p=float(top_p),
        )
        pieces.append(wav_t.detach().cpu().squeeze().numpy())
        if idx < len(chunks) - 1 and pause_sec > 0:
            pieces.append(np.zeros(int(model.sr * pause_sec), dtype=np.float32))

    full_wav = np.concatenate(pieces) if len(pieces) > 1 else pieces[0]
    out_path, wav_np = save_chatterbox_wav(full_wav, model.sr, prefix=f"{prefix}_{lang_code}")
    elapsed = time.time() - t0
    dur = len(wav_np) / model.sr
    return {
        "wav_path": out_path,
        "audio_np": wav_np,
        "sr": model.sr,
        "duration_s": dur,
        "elapsed_s": elapsed,
        "seed_used": seed_used,
        "language_id": lang_code,
        "chunks": len(chunks),
    }


def convert_voice_chatterbox(
    source_audio_path: str,
    target_voice_path: Optional[str] = None,
    prefix: str = "cb_vc",
) -> Dict[str, object]:
    """Run ChatterboxVC zero-shot audio-to-audio voice conversion."""
    if not source_audio_path or not os.path.exists(str(source_audio_path)):
        raise FileNotFoundError(f"Source audio file not found: {source_audio_path}")
    model = get_engine("vc")
    target_path = target_voice_path if (target_voice_path and os.path.exists(str(target_voice_path))) else None
    t0 = time.time()
    wav_t = model.generate(audio=str(source_audio_path), target_voice_path=target_path)
    out_path, wav_np = save_chatterbox_wav(wav_t, model.sr, prefix=prefix)
    elapsed = time.time() - t0
    dur = len(wav_np) / model.sr
    return {
        "wav_path": out_path,
        "audio_np": wav_np,
        "sr": model.sr,
        "duration_s": dur,
        "elapsed_s": elapsed,
    }


builtins.synthesize_expressive_tts = synthesize_expressive_tts
builtins.synthesize_turbo_tts = synthesize_turbo_tts
builtins.synthesize_multilingual_tts = synthesize_multilingual_tts
builtins.convert_voice_chatterbox = convert_voice_chatterbox
builtins.discover_aei_drive_audio = discover_aei_drive_audio
builtins.free_chatterbox_vram = free_chatterbox_vram
builtins.CHATTERBOX_OUT_DIR = OUT_DIR
builtins.CHATTERBOX_VOICES_DIR = VOICES_DIR

print("[Core] STEP 3 ready. Engines load lazily on first call.")
print(f"       Outputs saved to: {OUT_DIR}")

In [ ]:
# @title STEP 4 — Launch Gradio UI (4-Tab Chatterbox Voice Studio)
# @markdown Launches the interactive Gradio application with:
# @markdown 1. **Tab 1: Chatterbox TTS & Zero-Shot Voice Cloning** (`exaggeration` `0.0–2.0`, `cfg_weight` `0.0–1.0`, Cross-Notebook Drive Voice Picker)
# @markdown 2. **Tab 2: Chatterbox-Turbo (Fast 1-Step + Paralinguistic Tags) & Multilingual TTS (23 Languages)**
# @markdown 3. **Tab 3: Chatterbox Voice Conversion (`ChatterboxVC`)** (audio-to-audio timbre conversion)
# @markdown 4. **Tab 4: Batch Audiobook / Script Generator** (stitched master `.wav` + `.zip` bundle)

import os, time, zipfile
import numpy as np
import soundfile as sf
import gradio as gr
from tqdm.auto import tqdm
from IPython.display import clear_output

from chatterbox.mtl_tts import SUPPORTED_LANGUAGES

LANG_CHOICES = [f"{code} — {name}" for code, name in sorted(SUPPORTED_LANGUAGES.items())]


def _resolve_voice(uploaded_path, drive_choice):
    if uploaded_path and os.path.exists(str(uploaded_path)):
        return str(uploaded_path)
    if drive_choice and drive_choice != "(None — use built-in default voice)" and os.path.exists(str(drive_choice)):
        return str(drive_choice)
    return None


def _refresh_drive_choices():
    files = discover_aei_drive_audio(max_files=60)
    choices = ["(None — use built-in default voice)"] + files
    return gr.update(choices=choices, value=choices[0])


def _ui_tts(text, ref_upload, ref_drive, exaggeration, cfg_weight, temp, min_p, top_p, rep_pen, seed, auto_chunk, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error("Please enter text to synthesize.")
    voice_path = _resolve_voice(ref_upload, ref_drive)
    try:
        progress(0.15, desc="Synthesizing with ChatterboxTTS...")
        res = synthesize_expressive_tts(
            text=text,
            audio_prompt_path=voice_path,
            exaggeration=exaggeration,
            cfg_weight=cfg_weight,
            temperature=temp,
            min_p=min_p,
            top_p=top_p,
            repetition_penalty=rep_pen,
            seed=int(seed),
            auto_chunk=bool(auto_chunk),
            prefix="cb_expressive",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Engine: ChatterboxTTS (24 kHz) · Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s\n"
            f"Exaggeration: {exaggeration:.2f} · CFG/Pace: {cfg_weight:.2f} · Chunks: {res['chunks']} · Seed: {res['seed_used']}\n"
            f"Reference Voice: {os.path.basename(voice_path) if voice_path else 'Built-in Default'}\n"
            f"Saved to Drive: {res['wav_path']}"
        )
        return (res["sr"], res["audio_np"]), res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"ChatterboxTTS failed: {e}")


def _ui_turbo_or_mtl(mode, text, lang_choice, ref_upload, ref_drive, exaggeration, cfg_weight, temp, top_p, top_k, min_p, rep_pen, norm_loudness, seed, auto_chunk, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error("Please enter text to synthesize.")
    voice_path = _resolve_voice(ref_upload, ref_drive)
    try:
        if mode.startswith("Chatterbox-Turbo") or mode.startswith("Chatterbox-Nano"):
            is_nano = mode.startswith("Chatterbox-Nano")
            progress(0.15, desc=f"Synthesizing with {'Chatterbox-Nano' if is_nano else 'Chatterbox-Turbo'}...")
            res = synthesize_turbo_tts(
                text=text,
                audio_prompt_path=voice_path,
                nano=is_nano,
                temperature=temp,
                min_p=min_p,
                top_p=top_p,
                top_k=int(top_k),
                repetition_penalty=rep_pen,
                norm_loudness=bool(norm_loudness),
                seed=int(seed),
                auto_chunk=bool(auto_chunk),
                prefix="cb_nano" if is_nano else "cb_turbo",
            )
            info = (
                f"Engine: {'Chatterbox-Nano (110M)' if is_nano else 'Chatterbox-Turbo (350M 1-Step MeanFlow)'}\n"
                f"Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s ({res['duration_s']/max(res['elapsed_s'], 0.01):.1f}x RT) · Seed: {res['seed_used']}\n"
                f"Saved to Drive: {res['wav_path']}"
            )
            return (res["sr"], res["audio_np"]), res["wav_path"], info
        else:
            t3_ver = "v3" if "v3" in mode else "v2"
            lang_code = (lang_choice or "en").split("—")[0].strip()
            progress(0.15, desc=f"Synthesizing Multilingual ({lang_code}, {t3_ver})...")
            res = synthesize_multilingual_tts(
                text=text,
                language_id=lang_code,
                audio_prompt_path=voice_path,
                t3_version=t3_ver,
                exaggeration=exaggeration,
                cfg_weight=cfg_weight,
                temperature=temp,
                repetition_penalty=rep_pen,
                min_p=min_p,
                top_p=top_p,
                seed=int(seed),
                auto_chunk=bool(auto_chunk),
                prefix=f"cb_mtl_{t3_ver}",
            )
            info = (
                f"Engine: ChatterboxMultilingualTTS ({t3_ver}) · Language: {lang_choice}\n"
                f"Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s · Seed: {res['seed_used']}\n"
                f"Saved to Drive: {res['wav_path']}"
            )
            return (res["sr"], res["audio_np"]), res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"Generation failed: {e}")


def _ui_vc(src_upload, src_drive, target_upload, target_drive, progress=gr.Progress()):
    src_path = _resolve_voice(src_upload, src_drive)
    if not src_path:
        raise gr.Error("Upload or select a Source Audio clip from Drive to convert.")
    target_path = _resolve_voice(target_upload, target_drive)
    try:
        progress(0.2, desc="Converting voice timbre with ChatterboxVC...")
        res = convert_voice_chatterbox(source_audio_path=src_path, target_voice_path=target_path, prefix="cb_vc")
        progress(1.0, desc="Done!")
        info = (
            f"Engine: ChatterboxVC (S3Tokenizer -> S3Gen 24 kHz)\n"
            f"Source: {os.path.basename(src_path)} · Target: {os.path.basename(target_path) if target_path else 'Built-in Default'}\n"
            f"Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s\n"
            f"Saved to Drive: {res['wav_path']}"
        )
        return (res["sr"], res["audio_np"]), res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"Voice conversion failed: {e}")


def _ui_batch(script_text, engine_mode, lang_choice, ref_upload, ref_drive, exaggeration, cfg_weight, pause_sec, seed, progress=gr.Progress()):
    lines = [ln.strip() for ln in (script_text or "").splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not lines:
        raise gr.Error("Provide at least one non-empty line in the script box.")
    voice_path = _resolve_voice(ref_upload, ref_drive)
    lang_code = (lang_choice or "en").split("—")[0].strip()

    batch_ts = int(time.time())
    batch_dir = os.path.join(OUT_DIR, f"batch_{batch_ts}")
    os.makedirs(batch_dir, exist_ok=True)
    zip_path = os.path.join(OUT_DIR, f"chatterbox_batch_{batch_ts}.zip")
    stitched_pieces = []
    sr_out = 24000
    logs = []

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i, line in enumerate(tqdm(lines, desc="Batch Script")):
            progress((i + 1) / len(lines), desc=f"Synthesizing line {i+1}/{len(lines)}...")
            line_seed = (int(seed) + i) if int(seed) > 0 else 0
            try:
                if engine_mode == "Chatterbox-Turbo (Fast + Tags)":
                    r = synthesize_turbo_tts(line, audio_prompt_path=voice_path, seed=line_seed, prefix=f"line_{i+1:03d}")
                elif engine_mode == "Chatterbox-Multilingual (23 Langs)":
                    r = synthesize_multilingual_tts(line, language_id=lang_code, audio_prompt_path=voice_path, exaggeration=exaggeration, cfg_weight=cfg_weight, seed=line_seed, prefix=f"line_{i+1:03d}")
                else:
                    r = synthesize_expressive_tts(line, audio_prompt_path=voice_path, exaggeration=exaggeration, cfg_weight=cfg_weight, seed=line_seed, prefix=f"line_{i+1:03d}")
                sr_out = r["sr"]
                stitched_pieces.append(r["audio_np"])
                if i < len(lines) - 1 and float(pause_sec) > 0:
                    stitched_pieces.append(np.zeros(int(sr_out * float(pause_sec)), dtype=np.float32))
                arcname = f"{i+1:03d}_{os.path.basename(r['wav_path'])}"
                zf.write(r["wav_path"], arcname=arcname)
                logs.append(f"[{i+1:03d}/{len(lines)}] ✓ {r['duration_s']:.2f}s · {line[:65]}")
            except Exception as e:
                logs.append(f"[{i+1:03d}/{len(lines)}] ✗ ERROR: {e} · {line[:50]}")

    if not stitched_pieces:
        raise gr.Error("All lines failed to synthesize. Check the log for details.")
    master_wav = np.concatenate(stitched_pieces)
    master_path = os.path.join(OUT_DIR, f"audiobook_master_{batch_ts}.wav")
    sf.write(master_path, master_wav, sr_out, subtype="PCM_16")
    with zipfile.ZipFile(zip_path, "a", zipfile.ZIP_DEFLATED) as zf:
        zf.write(master_path, arcname=os.path.basename(master_path))
    logs.append(f"\n✓ Master Audiobook saved: {master_path} ({len(master_wav)/sr_out:.2f}s total)")
    return (sr_out, master_wav), master_path, zip_path, "\n".join(logs)


def _append_tag(current_text, tag):
    current_text = (current_text or "").rstrip()
    return f"{current_text} {tag} " if current_text else f"{tag} "


def _load_sample_lang_text(lang_choice):
    code = (lang_choice or "en").split("—")[0].strip()
    return MULTILINGUAL_SAMPLE_TEXTS.get(code, MULTILINGUAL_SAMPLE_TEXTS["en"])


initial_drive_choices = ["(None — use built-in default voice)"] + discover_aei_drive_audio(max_files=60)

with gr.Blocks(title="Chatterbox & Chatterbox-Turbo Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎙️ Chatterbox & Chatterbox-Turbo Voice Studio\n"
        "Zero-Shot Voice Cloning · Emotion Exaggeration (`0.0–2.0`) · 1-Step Turbo + Paralinguistic Tags · 23-Language Multilingual TTS · Voice Conversion (`ChatterboxVC`)"
    )
    status_banner = gr.Markdown()

    with gr.Tabs():
        # ── TAB 1: Expressive TTS & Zero-Shot Cloning ──
        with gr.Tab("1. Expressive TTS & Voice Cloning"):
            with gr.Row():
                with gr.Column(scale=3):
                    t1_text = gr.Textbox(
                        label="Text to Synthesize",
                        lines=5,
                        value="Now let's make my mum's favourite recipe! Three Mars bars into the pan, a sprinkle of olive oil, and oh boy, this is going to be incredible.",
                        info="Paragraphs over 280 characters are automatically split on sentence boundaries when Auto-Chunk is enabled.",
                    )
                    t1_ref_up = gr.Audio(
                        sources=["upload", "microphone"],
                        type="filepath",
                        label="Reference Voice Audio (5–15s clean speech, optional)",
                    )
                    with gr.Row():
                        t1_ref_drive = gr.Dropdown(
                            choices=initial_drive_choices,
                            value=initial_drive_choices[0],
                            label="📂 Or Pick Reference Voice from Google Drive (AEI_TTS_Cache / AEI_TTS_Out)",
                            info="Cross-notebook picker: discovers voices & outputs saved on Google Drive.",
                        )
                        t1_refresh_btn = gr.Button("🔄 Refresh Drive", scale=0)
                with gr.Column(scale=2):
                    t1_exag = gr.Slider(
                        0.0, 2.0, value=0.5, step=0.05,
                        label="Emotion Exaggeration",
                        info="0.5 = Neutral, 0.7–1.2 = Expressive/Dramatic, <0.3 = Calm/Subdued.",
                    )
                    t1_cfg = gr.Slider(
                        0.0, 1.0, value=0.5, step=0.05,
                        label="CFG Weight / Pace Adherence",
                        info="Default 0.5. Lower (~0.3) slows down fast reference speakers; higher increases prompt adherence.",
                    )
                    with gr.Accordion("Advanced Sampling & Long-Form Options", open=False):
                        t1_temp = gr.Slider(0.05, 2.0, value=0.8, step=0.05, label="Temperature", info="Controls sampling randomness (default 0.8).")
                        t1_min_p = gr.Slider(0.0, 1.0, value=0.05, step=0.01, label="Min-P", info="Newer sampler cutoff (recommended 0.02–0.10).")
                        t1_top_p = gr.Slider(0.0, 1.0, value=1.0, step=0.01, label="Top-P", info="Nucleus sampling (1.0 disables when using Min-P).")
                        t1_rep = gr.Slider(1.0, 2.0, value=1.2, step=0.05, label="Repetition Penalty", info="Penalizes repeated speech tokens.")
                        t1_seed = gr.Number(value=0, precision=0, label="Seed (0 = random)", info="Set >0 for reproducible output.")
                        t1_chunk = gr.Checkbox(value=True, label="Auto-Chunk Long Text by Sentence", info="Splits long passages cleanly and stitches with natural pauses.")
                    t1_btn = gr.Button("🎙️ Synthesize Expressive Speech", variant="primary")
                    t1_out_audio = gr.Audio(label="Synthesized Output (24 kHz)", type="numpy")
                    t1_out_file = gr.File(label="Download WAV")
                    t1_out_info = gr.Textbox(label="Synthesis Metadata & Drive Path", lines=4, interactive=False)

            t1_refresh_btn.click(_refresh_drive_choices, outputs=[t1_ref_drive])
            t1_btn.click(
                _ui_tts,
                inputs=[t1_text, t1_ref_up, t1_ref_drive, t1_exag, t1_cfg, t1_temp, t1_min_p, t1_top_p, t1_rep, t1_seed, t1_chunk],
                outputs=[t1_out_audio, t1_out_file, t1_out_info],
            )

        # ── TAB 2: Chatterbox-Turbo (Paralinguistic Tags) & Multilingual TTS ──
        with gr.Tab("2. Chatterbox-Turbo (Tags) & Multilingual (23 Langs)"):
            with gr.Row():
                with gr.Column(scale=3):
                    t2_mode = gr.Radio(
                        choices=[
                            "Chatterbox-Turbo (Fast 1-Step + Paralinguistic Tags)",
                            "Chatterbox-Nano (Ultra-Light 110M + Paralinguistic Tags)",
                            "Chatterbox-Multilingual v2 (23 Languages)",
                            "Chatterbox-Multilingual v3 (23 Languages)",
                        ],
                        value="Chatterbox-Turbo (Fast 1-Step + Paralinguistic Tags)",
                        label="Engine Variant",
                        info="Turbo/Nano support paralinguistic event tags like [laugh] and [chuckle]. Multilingual supports 23 languages.",
                    )
                    t2_text = gr.Textbox(
                        label="Text to Synthesize (click tag buttons below for Turbo/Nano)",
                        lines=5,
                        value="Oh, that's hilarious! [chuckle] Um anyway, we just launched our new voice model and [sigh] it sounds ridiculously natural!",
                        info="Insert paralinguistic tags anywhere in English text when using Chatterbox-Turbo or Nano.",
                    )
                    gr.Markdown("**Quick-Insert Paralinguistic Tags (for Chatterbox-Turbo / Nano):**")
                    with gr.Row():
                        for _tag in TURBO_EVENT_TAGS:
                            _b = gr.Button(_tag, size="sm")
                            _b.click(fn=lambda txt, tg=_tag: _append_tag(txt, tg), inputs=[t2_text], outputs=[t2_text])
                    with gr.Row():
                        t2_lang = gr.Dropdown(
                            choices=LANG_CHOICES,
                            value="en — English",
                            label="🌍 Target Language (used when Multilingual v2/v3 is selected)",
                            info="23 languages supported by ChatterboxMultilingualTTS.",
                        )
                        t2_sample_btn = gr.Button("📝 Load Sample Text for Language", scale=0)
                    t2_ref_up = gr.Audio(
                        sources=["upload", "microphone"],
                        type="filepath",
                        label="Reference Voice Audio (optional, zero-shot cross-lingual cloning)",
                    )
                    t2_ref_drive = gr.Dropdown(
                        choices=initial_drive_choices,
                        value=initial_drive_choices[0],
                        label="📂 Or Pick Reference Voice from Google Drive",
                        info="Select any saved voice prompt from AEI_TTS_Cache/voices or AEI_TTS_Out.",
                    )
                with gr.Column(scale=2):
                    t2_exag = gr.Slider(0.0, 2.0, value=0.5, step=0.05, label="Emotion Exaggeration (Multilingual mode)", info="Used by Multilingual v2/v3.")
                    t2_cfg = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="CFG Weight (Multilingual mode)", info="Used by Multilingual v2/v3.")
                    t2_loudness = gr.Checkbox(value=True, label="Normalize Loudness (-27 LUFS, Turbo/Nano)", info="Applies ITU-R BS.1770 loudness normalization.")
                    with gr.Accordion("Sampling Parameters", open=False):
                        t2_temp = gr.Slider(0.05, 2.0, value=0.8, step=0.05, label="Temperature", info="Sampling temperature (default 0.8).")
                        t2_top_p = gr.Slider(0.0, 1.0, value=0.95, step=0.01, label="Top-P", info="Default 0.95 for Turbo.")
                        t2_top_k = gr.Slider(0, 1000, value=1000, step=10, label="Top-K (Turbo/Nano)", info="Default 1000 for Turbo.")
                        t2_min_p = gr.Slider(0.0, 1.0, value=0.0, step=0.01, label="Min-P", info="0.0 disables for Turbo; 0.05 for Multilingual.")
                        t2_rep = gr.Slider(1.0, 2.0, value=1.2, step=0.05, label="Repetition Penalty", info="Default 1.2.")
                        t2_seed = gr.Number(value=0, precision=0, label="Seed (0 = random)", info="Deterministic seed when >0.")
                        t2_chunk = gr.Checkbox(value=True, label="Auto-Chunk Long Text", info="Splits multi-sentence paragraphs cleanly.")
                    t2_btn = gr.Button("⚡ Generate Turbo / Multilingual Speech", variant="primary")
                    t2_out_audio = gr.Audio(label="Output Audio (24 kHz)", type="numpy")
                    t2_out_file = gr.File(label="Download WAV")
                    t2_out_info = gr.Textbox(label="Run Summary", lines=4, interactive=False)

            t2_sample_btn.click(_load_sample_lang_text, inputs=[t2_lang], outputs=[t2_text])
            t2_btn.click(
                _ui_turbo_or_mtl,
                inputs=[t2_mode, t2_text, t2_lang, t2_ref_up, t2_ref_drive, t2_exag, t2_cfg, t2_temp, t2_top_p, t2_top_k, t2_min_p, t2_rep, t2_loudness, t2_seed, t2_chunk],
                outputs=[t2_out_audio, t2_out_file, t2_out_info],
            )

        # ── TAB 3: Chatterbox Voice Conversion (ChatterboxVC) ──
        with gr.Tab("3. Voice Conversion (ChatterboxVC)"):
            gr.Markdown("Convert any spoken audio clip into the timbre of a target reference voice using **`ChatterboxVC`** (`S3Tokenizer` $\rightarrow$ `S3Gen`).")
            with gr.Row():
                with gr.Column():
                    t3_src_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Source Speech Audio (content & timing to keep)")
                    t3_src_drive = gr.Dropdown(
                        choices=initial_drive_choices,
                        value=initial_drive_choices[0],
                        label="📂 Or Pick Source Audio from Google Drive",
                        info="Grab speech generated by any other AEI TTS notebook.",
                    )
                    t3_tgt_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Target Reference Voice (timbre to clone)")
                    t3_tgt_drive = gr.Dropdown(
                        choices=initial_drive_choices,
                        value=initial_drive_choices[0],
                        label="📂 Or Pick Target Voice from Google Drive",
                        info="Leave empty to use Chatterbox's built-in voice.",
                    )
                    t3_btn = gr.Button("🔄 Convert Voice Timbre", variant="primary")
                with gr.Column():
                    t3_out_audio = gr.Audio(label="Converted Output Audio (24 kHz)", type="numpy")
                    t3_out_file = gr.File(label="Download Converted WAV")
                    t3_out_info = gr.Textbox(label="Conversion Info", lines=4, interactive=False)

            t3_btn.click(
                _ui_vc,
                inputs=[t3_src_up, t3_src_drive, t3_tgt_up, t3_tgt_drive],
                outputs=[t3_out_audio, t3_out_file, t3_out_info],
            )

        # ── TAB 4: Batch Audiobook / Script Generator ──
        with gr.Tab("4. Batch Audiobook / Script Generator"):
            with gr.Row():
                with gr.Column(scale=3):
                    t4_script = gr.Textbox(
                        label="Multi-Line Script (one line/paragraph per utterance)",
                        lines=9,
                        value=(
                            "Chapter One. The neon lights of Neo-Tokyo flickered against the rain-slicked glass.\n"
                            "She leaned closer to the terminal and whispered, [sigh] We only get one shot at this.\n"
                            "Wait, did you hear that? [chuckle] Our backup system actually worked!"
                        ),
                        info="Each line is synthesized individually and stitched into a master audiobook WAV + ZIP archive.",
                    )
                    with gr.Row():
                        t4_engine = gr.Dropdown(
                            choices=[
                                "ChatterboxTTS (Expressive Emotion)",
                                "Chatterbox-Turbo (Fast + Tags)",
                                "Chatterbox-Multilingual (23 Langs)",
                            ],
                            value="Chatterbox-Turbo (Fast + Tags)",
                            label="Batch Engine",
                            info="Select which Chatterbox engine renders the script.",
                        )
                        t4_lang = gr.Dropdown(
                            choices=LANG_CHOICES,
                            value="en — English",
                            label="Language (if Multilingual)",
                            info="Language used when Chatterbox-Multilingual is selected.",
                        )
                    t4_ref_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Reference Voice for Script (optional)")
                    t4_ref_drive = gr.Dropdown(
                        choices=initial_drive_choices,
                        value=initial_drive_choices[0],
                        label="📂 Or Pick Reference Voice from Google Drive",
                        info="Consistent reference voice across all lines.",
                    )
                with gr.Column(scale=2):
                    t4_exag = gr.Slider(0.0, 2.0, value=0.5, step=0.05, label="Exaggeration", info="Used by Expressive & Multilingual engines.")
                    t4_cfg = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="CFG Weight", info="Pacing & prompt adherence.")
                    t4_pause = gr.Slider(0.0, 2.0, value=0.35, step=0.05, label="Pause Between Lines (seconds)", info="Silence inserted between consecutive script lines.")
                    t4_seed = gr.Number(value=42, precision=0, label="Base Seed (0 = random)", info="Incremented per line when >0.")
                    t4_btn = gr.Button("📚 Render Batch Audiobook & ZIP", variant="primary")
                    t4_out_audio = gr.Audio(label="Stitched Master Audiobook", type="numpy")
                    t4_out_wav = gr.File(label="Download Master WAV")
                    t4_out_zip = gr.File(label="Download All Lines (.zip)")
                    t4_log = gr.Textbox(label="Batch Execution Log", lines=6, interactive=False)

            t4_btn.click(
                _ui_batch,
                inputs=[t4_script, t4_engine, t4_lang, t4_ref_up, t4_ref_drive, t4_exag, t4_cfg, t4_pause, t4_seed],
                outputs=[t4_out_audio, t4_out_wav, t4_out_zip, t4_log],
            )

    with gr.Accordion("🧹 GPU VRAM Manager", open=False):
        vram_box = gr.Textbox(label="VRAM Status", interactive=False)
        free_btn = gr.Button("Unload Cached Engines & Free VRAM")
        free_btn.click(fn=lambda: free_chatterbox_vram(), outputs=[vram_box])

    def _on_load():
        return (
            f"✅ **Chatterbox Studio Ready** on `{DEVICE}` · Outputs auto-saved to `{OUT_DIR}` "
            f"(instantly accessible in `InfiniteTalk` & `LongCat-Video-Avatar-1.5`).",
            free_chatterbox_vram(),
        )

    demo.load(_on_load, inputs=None, outputs=[status_banner, vram_box])

clear_output()
demo.queue(concurrency_limit=1).launch(share=True, inline=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep Colab Session Alive
# @markdown Starts a lightweight background heartbeat thread so Google Colab does not disconnect while you use the Gradio UI or run long audiobook batches.

import threading, time

def _chatterbox_keepalive():
    while True:
        time.sleep(60)

threading.Thread(target=_chatterbox_keepalive, daemon=True).start()
print("[Keep-Alive] Heartbeat thread active. Outputs dir:", OUT_DIR)

In [ ]:
# @title STEP 6 — Quick Test (Form-Driven Synthesis without UI)
# @markdown Run a single synthesis or voice conversion call directly from form parameters and preview/download the resulting `.wav`.

ENGINE_MODE = "turbo"  # @param ["tts", "turbo", "nano", "multilingual_v2", "multilingual_v3", "vc"]
TEXT = "Oh, that's hilarious! [chuckle] Welcome to the Chatterbox voice cloning studio on Google Colab."  # @param {type:"string"}
LANGUAGE_ID = "en"  # @param ["ar", "da", "de", "el", "en", "es", "fi", "fr", "he", "hi", "it", "ja", "ko", "ms", "nl", "no", "pl", "pt", "ru", "sv", "sw", "tr", "zh"]
REFERENCE_VOICE_PATH = ""  # @param {type:"string"}
SOURCE_AUDIO_FOR_VC = ""  # @param {type:"string"}
EXAGGERATION = 0.5  # @param {type:"slider", min:0.0, max:2.0, step:0.05}
CFG_WEIGHT = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
TEMPERATURE = 0.8  # @param {type:"slider", min:0.1, max:1.5, step:0.05}
SEED = 42  # @param {type:"integer"}

import os
from IPython.display import Audio, FileLink, display

ref_path = REFERENCE_VOICE_PATH.strip() or None
if ref_path is None:
    cand = os.path.join(VOICES_DIR, "chatterbox_female_podcast.wav")
    if os.path.exists(cand):
        ref_path = cand

if ENGINE_MODE == "vc":
    src_vc = SOURCE_AUDIO_FOR_VC.strip() or ref_path
    if not src_vc or not os.path.exists(src_vc):
        raise SystemExit("Provide SOURCE_AUDIO_FOR_VC (or run Step 2 to cache sample audio) for VC mode.")
    res = convert_voice_chatterbox(src_vc, target_voice_path=ref_path, prefix="quicktest_vc")
elif ENGINE_MODE in ("turbo", "nano"):
    res = synthesize_turbo_tts(
        TEXT, audio_prompt_path=ref_path, nano=(ENGINE_MODE == "nano"),
        temperature=TEMPERATURE, seed=SEED, prefix=f"quicktest_{ENGINE_MODE}",
    )
elif ENGINE_MODE.startswith("multilingual"):
    t3_v = "v3" if ENGINE_MODE.endswith("v3") else "v2"
    res = synthesize_multilingual_tts(
        TEXT, language_id=LANGUAGE_ID, audio_prompt_path=ref_path, t3_version=t3_v,
        exaggeration=EXAGGERATION, cfg_weight=CFG_WEIGHT, temperature=TEMPERATURE,
        seed=SEED, prefix=f"quicktest_{ENGINE_MODE}",
    )
else:
    res = synthesize_expressive_tts(
        TEXT, audio_prompt_path=ref_path, exaggeration=EXAGGERATION,
        cfg_weight=CFG_WEIGHT, temperature=TEMPERATURE, seed=SEED, prefix="quicktest_tts",
    )

print(f"✓ Saved: {res['wav_path']} ({res['duration_s']:.2f}s @ {res['sr']} Hz in {res['elapsed_s']:.1f}s)")
display(Audio(res["wav_path"], autoplay=False))
display(FileLink(res["wav_path"]))

In [ ]:
# @title STEP 7 — Batch Script / Audiobook Generation (Fault-Tolerant with tqdm)
# @markdown Synthesizes a list of lines (from a `.txt`/`.json` file or default starter lines) with per-item `try`/`except` error handling and `tqdm` progress, saving both individual `.wav` files and a concatenated master audiobook `.wav` to `/content/drive/MyDrive/AEI_TTS_Out/Chatterbox/`.

BATCH_ENGINE = "turbo"  # @param ["tts", "turbo", "multilingual_v2"]
LANGUAGE_ID = "en"  # @param {type:"string"}
REFERENCE_VOICE_PATH = ""  # @param {type:"string"}
SCRIPT_FILE_PATH = ""  # @param {type:"string"}
EXAGGERATION = 0.6  # @param {type:"slider", min:0.0, max:2.0, step:0.05}
CFG_WEIGHT = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
PAUSE_SECONDS = 0.35  # @param {type:"slider", min:0.0, max:2.0, step:0.05}
BASE_SEED = 100  # @param {type:"integer"}

import os, time, json
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm
from IPython.display import Audio, FileLink, display

if SCRIPT_FILE_PATH.strip() and os.path.exists(SCRIPT_FILE_PATH.strip()):
    if SCRIPT_FILE_PATH.strip().endswith(".json"):
        raw = json.loads(open(SCRIPT_FILE_PATH.strip(), encoding="utf-8").read())
        batch_lines = [x["text"] if isinstance(x, dict) else str(x) for x in raw]
    else:
        batch_lines = [ln.strip() for ln in open(SCRIPT_FILE_PATH.strip(), encoding="utf-8") if ln.strip()]
else:
    batch_lines = [
        "Welcome to the automated batch synthesis pipeline powered by Resemble AI Chatterbox.",
        "Wait until you hear how natural paralinguistic expressions sound! [chuckle] Everything is saved directly to Google Drive.",
        "You can now load this audio directly inside InfiniteTalk or LongCat-Video-Avatar to animate talking avatars.",
    ]

ref_path = REFERENCE_VOICE_PATH.strip() or None
if ref_path is None:
    default_v = os.path.join(VOICES_DIR, "chatterbox_female_podcast.wav")
    if os.path.exists(default_v):
        ref_path = default_v

print(f"[Batch] Processing {len(batch_lines)} item(s) with engine={BATCH_ENGINE} ...")
stitched_audio = []
sr_out = 24000
ok_count = 0

for idx, line_text in enumerate(tqdm(batch_lines, desc="Step 7 Batch")):
    item_seed = (BASE_SEED + idx) if BASE_SEED > 0 else 0
    try:
        if BATCH_ENGINE == "turbo":
            r = synthesize_turbo_tts(line_text, audio_prompt_path=ref_path, seed=item_seed, prefix=f"batch_{idx+1:03d}")
        elif BATCH_ENGINE == "multilingual_v2":
            r = synthesize_multilingual_tts(
                line_text, language_id=LANGUAGE_ID, audio_prompt_path=ref_path,
                exaggeration=EXAGGERATION, cfg_weight=CFG_WEIGHT, seed=item_seed, prefix=f"batch_{idx+1:03d}",
            )
        else:
            r = synthesize_expressive_tts(
                line_text, audio_prompt_path=ref_path, exaggeration=EXAGGERATION,
                cfg_weight=CFG_WEIGHT, seed=item_seed, prefix=f"batch_{idx+1:03d}",
            )
        sr_out = r["sr"]
        stitched_audio.append(r["audio_np"])
        if idx < len(batch_lines) - 1 and PAUSE_SECONDS > 0:
            stitched_audio.append(np.zeros(int(sr_out * PAUSE_SECONDS), dtype=np.float32))
        ok_count += 1
        print(f"  [{idx+1:03d}/{len(batch_lines)}] ✓ {r['duration_s']:.2f}s -> {r['wav_path']}")
    except Exception as e:
        print(f"  [{idx+1:03d}/{len(batch_lines)}] ✗ FAILED: {e}")

if stitched_audio:
    master_np = np.concatenate(stitched_audio)
    master_out = os.path.join(OUT_DIR, f"batch_master_{int(time.time())}.wav")
    sf.write(master_out, master_np, sr_out, subtype="PCM_16")
    print(f"\n[Done] {ok_count}/{len(batch_lines)} succeeded · Master WAV: {master_out} ({len(master_np)/sr_out:.2f}s)")
    display(Audio(master_out, autoplay=False))
    display(FileLink(master_out))